# Day 2 · 02 · Performance, Lakeflow Jobs and DABs for BI Products

![Lakeflow Job DAG](../../assets/images/lakeflow_job_dag.png)

This notebook turns the Databricks + Power BI dataset into an operational BI
product: performance checks, cost guardrails, refresh strategy, Lakeflow Jobs
and DABs orientation.

## Business Scenario

The dashboard now has trusted data, but production concerns remain:

- some report pages are slow,
- DirectQuery can multiply SQL Warehouse usage,
- refresh must be repeatable,
- the job definition should not live only in the UI.

## Learning Objectives

By the end of this notebook you can:

- compare detail and aggregate query patterns,
- read basic performance symptoms,
- define BI cost guardrails,
- design a Lakeflow Job DAG,
- explain why DABs make deployment repeatable,
- complete a report certification checklist.

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this notebook.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before the demo starts.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream setup or demo notebook has not created the required tables.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
    f"{GOLD}.kpi_monthly",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: notebooks/demo/day2_01_powerbi_semantic_model.ipynb")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"notebooks/demo/day2_01_powerbi_semantic_model.ipynb\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## 1. Performance before/after

![Query profile reading map](../../assets/images/query_profile_reading_map.png)

Compare detail-grain and aggregate-grain queries for the same business
question.

### What Photon does (and does not) explain

Most SQL Warehouses in this course run on **Photon**, Databricks' vectorized
query engine. Before reading the plans below, be clear about what it does:

| Photon | Reality |
|---|---|
| Speeds up scans, filters, joins and aggregations | yes — vectorized execution processes batches of rows instead of one at a time |
| Helps most on | wide scans, heavy aggregation, large joins — exactly the kind of query a BI dashboard sends |
| Uses Predictive I/O | yes — smarter file/row skipping on top of Delta statistics |
| Fixes a bad data model | **no** — a query against an unfiltered detail table is still a big scan, just a faster big scan |
| Replaces partition pruning / good filters | **no** — Photon executes the plan faster, it does not rewrite a missing `WHERE` clause into one |
| Something you configure per query | no — it is a property of the warehouse/cluster; your job as an analyst is to write SQL that gives it a small, well-pruned plan to execute |

The takeaway for this section: Photon makes both queries below faster than
without it, but it cannot make the detail-table query as cheap as the
aggregate-table query. The plan shape still matters more than the engine.

In [ ]:
spark.sql(f"""
EXPLAIN FORMATTED
SELECT category, ROUND(SUM(line_revenue), 2) AS revenue
FROM {GOLD}.fact_sales_dashboard
WHERE is_completed
GROUP BY category
ORDER BY revenue DESC
""").show(truncate=False)

In [ ]:
spark.sql(f"""
EXPLAIN FORMATTED
SELECT category, ROUND(SUM(revenue), 2) AS revenue
FROM {GOLD}.fact_sales_dashboard_monthly
GROUP BY category
ORDER BY revenue DESC
""").show(truncate=False)

Discussion:

- Which query scans more rows?
- Which one should back a summary page?
- Which one is acceptable for a drill-through page?
- What happens if this is DirectQuery and every visual runs it repeatedly?

## 2. Query Insights — slow BI visual investigation

This is the most useful operational skill for a Databricks analyst: given a
slow report page, find which query caused it, why it was expensive, and how
to fix it. The next cells simulate that investigation end to end.

| Tool | What it shows | Use it for |
|---|---|---|
| Query History | every query: user, warehouse, status, start time, duration | "who ran what, when, and did it finish" |
| Query Profile | visual DAG for one query: scan, filter, join, shuffle, aggregate nodes, bytes/rows at each step | "where did the time and the bytes actually go" |
| Filtering query history | by user, warehouse, status, duration | finding the worst offenders without scrolling through everything |

### Step 1 — run a deliberately broad query

This is the kind of query an unfiltered BI visual or a careless `SELECT *`
report page sends.

In [ ]:
spark.sql(f"""
SELECT *
FROM {GOLD}.fact_sales_dashboard
ORDER BY order_date DESC
""").show(5)

### Step 2 — open Query History and Query Profile

`[UI DEMO]` Open **Query History** for the query above and check:

- duration and status,
- which warehouse ran it,
- rows produced vs rows read.

Then open its **Query Profile** and look for:

- the `Scan` node: bytes/rows scanned, columns read,
- whether any filter was pushed down to the scan,
- whether the whole table was read because there was no `WHERE` clause,
- the overall time breakdown across nodes.

### Step 3 — diagnose

For this query, the likely findings are:

- `SELECT *` reads every column, including ones the report never displays,
- no `WHERE` clause means no partition pruning and no predicate pushdown,
- `ORDER BY` without `LIMIT` upstream of the visual forces a full sort,
- this is a detail-grain table — the same shape problem from Section 1.

### Step 4 — rewrite

In [ ]:
spark.sql(f"""
SELECT year_month, category, ROUND(SUM(revenue), 2) AS revenue
FROM {GOLD}.fact_sales_dashboard_monthly
WHERE year_month = (SELECT MAX(year_month) FROM {GOLD}.fact_sales_dashboard_monthly)
GROUP BY year_month, category
ORDER BY revenue DESC
""").show()

### Step 5 — compare before/after

`[UI DEMO]` Open Query History again and compare the two queries:

- explicit columns instead of `SELECT *`,
- a filter that narrows the scan instead of reading every row,
- an aggregate table instead of a detail table,
- a bounded result instead of an unbounded sort.

Same investigation pattern works for any slow visual: Query History to find
it, Query Profile to see why, then rewrite and re-check.

## 3. BI SQL Best Practices checklist

A short reference, not a full optimization lab. Each row is something to check
before a query backs a BI visual.

| Anti-pattern | Better pattern | Why |
|---|---|---|
| `SELECT *` for a BI visual | select only the columns the visual needs | less scan, less network transfer to Power BI |
| filter applied after a join | filter the fact table first, then join dimensions | smaller intermediate data for the join/shuffle |
| DirectQuery against a detail/line-grain table | DirectQuery against an aggregate or a narrow view | lower latency per visual interaction, lower warehouse cost |
| every dashboard refresh re-aggregates from detail | a materialized aggregate table (like `fact_sales_dashboard_monthly`) | predictable, bounded refresh cost instead of repeated full scans |
| random or high-cardinality partitioning | partition or cluster by a useful, low-cardinality, date-like column | avoids tiny partitions and wasted file listing |
| `ZORDER BY` on every new table | **Liquid Clustering** for new large tables with evolving filters | modern default; adapts without a full rewrite like `ZORDER` requires |
| Python UDF for row-level SQL logic | SQL UDF when the workload is SQL/BI | SQL UDFs can be optimized inside the SQL engine; Python UDFs cannot |

This checklist is the BI-facing summary of the deeper optimization topics in
`m4f_best_practices_sql_analytics.ipynb` (full course) — `OPTIMIZE`, `ZORDER`
mechanics and `ANALYZE TABLE ... COMPUTE STATISTICS` are bonus/reference
material, not required for this course.

## 4. Cost guardrails

Use `docs/templates/cost-awareness-checklist.md`.

| Area | Guardrail |
|---|---|
| Warehouse size | start small, scale only with evidence |
| Auto-stop | 5-10 minutes for training/demo |
| Import mode | default for executive dashboard |
| DirectQuery | exception, only on Gold objects |
| Aggregates | summary visuals use aggregate tables |
| Monitoring | review Query History and billing/system tables |

## 5. Refresh strategy

Use `docs/templates/refresh-strategy.md`.

Recommended BI refresh flow:

1. validate Silver sources,
2. refresh Gold dimensions and facts,
3. build KPI and BI aggregates,
4. validate reconciliation,
5. refresh Power BI dataset or expose updated Gold objects,
6. record certification status.

## 6. Lakeflow Jobs orientation

Lakeflow Jobs turn the BI refresh path into an observable, repeatable workflow.
For this course, focus on why each task exists, what should fail fast, and
which outputs Power BI depends on.

| Task | Purpose | Failure behavior |
|---|---|---|
| `validate_sources` | confirm required Silver tables and bad-data expectations | fail fast |
| `refresh_gold` | build dimensions, fact and dashboard table | retry once |
| `build_bi_objects` | build KPI, aggregates and incremental view | retry once |
| `validate_bi_readiness` | reconciliation and certification checks | fail and notify owner |

`[UI DEMO]` Show Jobs UI: task graph, run details, logs, repair run and
parameters.

## 7. DABs / Declarative Automation Bundles

![DABs deployment flow](../../assets/images/dabs_deployment_flow.png)

DABs answers a simple operational question: can we recreate the job in another
environment from code instead of clicking through the UI again?

The reference file is `bundle/databricks.yml`.

In [ ]:
# Reference-only: inspect the generated bundle file from the repo.
# In a Databricks workspace repo, this path may differ. Treat this cell as
# documentation when running outside the local repo context.
print("Reference DABs file: bundle/databricks.yml")
print("CLI commands to run outside this notebook:")
print("  databricks bundle validate -t dev")
print("  databricks bundle deploy -t dev")
print("  databricks bundle run refresh_gold_bi_dataset -t dev")

## 8. Report certification checklist

Use `docs/templates/report-certification-checklist.md`.

| Check | Required evidence |
|---|---|
| KPI approved | KPI dictionary filled |
| Grain known | fact and aggregate grain documented |
| Source object chosen | BI contract filled |
| Refresh defined | refresh strategy filled |
| Cost guardrails defined | checklist filled |
| Reconciliation passed | max diff within tolerance |
| Owner named | business and technical owner |

In [ ]:
recon = spark.sql(f"""
WITH agg AS (
  SELECT year_month, ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_monthly
  GROUP BY year_month
),
kpi AS (
  SELECT year_month, ROUND(revenue, 2) AS revenue
  FROM {GOLD}.kpi_monthly
)
SELECT MAX(ABS(agg.revenue - kpi.revenue)) AS max_diff
FROM agg JOIN kpi ON agg.year_month = kpi.year_month
""").first()["max_diff"]

assert recon < 0.01, f"Certification failed: revenue reconciliation diff {recon}"
print("[OK] Certification reconciliation passed")

## 9. Capstone

Final package participants should be able to explain:

- Gold KPI package,
- Power BI semantic dataset,
- Import vs DirectQuery decision,
- performance risk and mitigation (Photon, query plans, Query Insights),
- BI SQL best practices applied,
- Lakeflow Job DAG,
- DABs deployment story,
- certification checklist.

This is the handoff from a notebook exercise to a real BI product pattern.

## Summary

After this notebook, the training has covered the full loop:

1. explore data and warehouse cost,
2. build Gold,
3. define KPI,
4. prepare Power BI,
5. validate performance,
6. automate refresh,
7. document and certify the report.